[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ml-matthew-lam/relightable-3dgs/blob/main/training.ipynb)

# **Relightable 3DGS Training Notebook**

## Checking the GPU / CUDA version

In [5]:
!nvidia-smi

## Mounting Google Drive

In [6]:
from google.colab import drive
drive.mount('/content/drive')

## Cloning repo and installing dependencies

In [25]:
import os

os.chdir("/content")  # always start from a fixed base, so this cell is safe to rerun in a live session
if not os.path.exists("/content/relightable-3dgs"):
    !git clone https://github.com/ml-matthew-lam/relightable-3dgs.git
%cd /content/relightable-3dgs
!git pull

In [8]:
!pip install -r requirements.txt

Restart runtime now, in case torch was somehow imported before installing `requirements.txt`.

## Copying dataset from Drive to Colab's local disk

In [9]:
import os
import shutil

DRIVE_DATASET_ZIP = "/content/drive/MyDrive/Personal Projects/relightable-3dgs/checkered_suzanne.zip"
LOCAL_DATASET_PATH = "/content/checkered_suzanne"

if not os.path.exists(LOCAL_DATASET_PATH):
    local_zip = "/content/checkered_suzanne.zip"
    shutil.copy(DRIVE_DATASET_ZIP, local_zip)
    shutil.unpack_archive(local_zip, "/content")
    os.remove(local_zip)
    print(f"copied and unzipped dataset to {LOCAL_DATASET_PATH}")
else:
    print(f"{LOCAL_DATASET_PATH} already exists, skipped copy")

## Running the training script

In [ ]:
DRIVE_CHECKPOINT_DIR = "/content/drive/MyDrive/Personal Projects/relightable-3dgs/checkpoints/lambertian_week3"

!python train.py \
  --data_dir /content/checkered_suzanne \
  --ckpt_dir "{DRIVE_CHECKPOINT_DIR}" \
  --iters 15000 \
  --num_init_points 100000

## Resuming in case of a disconnect
If runtime disconnects mid-run: reconnect, re-run cells 1-4, then run the cell below instead of the cell above.

In [ ]:
DRIVE_CHECKPOINT_DIR = "/content/drive/MyDrive/Personal Projects/relightable-3dgs/checkpoints/lambertian_week3"

!python train.py \
  --data_dir /content/checkered_suzanne \
  --ckpt_dir "{DRIVE_CHECKPOINT_DIR}" \
  --iters 15000 \
  --num_init_points 100000 \
  --resume

## Comparing renders

In [ ]:
# BEAUTY
!python render_compare.py \
  --data_dir /content/checkered_suzanne \
  --ckpt "{DRIVE_CHECKPOINT_DIR}/step_015000.pt" \
  --render_type beauty \
  --out_dir "/content/drive/MyDrive/Personal Projects/relightable-3dgs/comparisons/week3_beauty"

In [ ]:
# NORMALS
!python render_compare.py \
  --data_dir /content/checkered_suzanne \
  --ckpt "{DRIVE_CHECKPOINT_DIR}/step_015000.pt" \
  --render_type normals \
  --out_dir "/content/drive/MyDrive/Personal Projects/relightable-3dgs/comparisons/week3_normals"

In [ ]:
# ALBEDO
!python render_compare.py \
  --data_dir /content/checkered_suzanne \
  --ckpt "{DRIVE_CHECKPOINT_DIR}/step_015000.pt" \
  --render_type albedo \
  --out_dir "/content/drive/MyDrive/Personal Projects/relightable-3dgs/comparisons/week3_albedo"

In [ ]:
# BEAUTY - test light (light5)
!python render_compare.py \
  --data_dir /content/checkered_suzanne \
  --ckpt "{DRIVE_CHECKPOINT_DIR}/step_015000.pt" \
  --render_type beauty \
  --test_light \
  --out_dir "/content/drive/MyDrive/Personal Projects/relightable-3dgs/comparisons/week3_test_beauty"

## Checking that roughness values did not change

In [33]:
import torch

ckpt = torch.load(f"{DRIVE_CHECKPOINT_DIR}/step_015000.pt", map_location="cpu")
roughness = ckpt["params"]["roughness_logits"]
print(roughness.unique())